In [12]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
import ast
from googletrans import Translator
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPTokenizer, CLIPTextModel
from openai import OpenAI
import json
import re
import anthropic
import torch
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
import copy

In [2]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [3]:
artnet_2024.columns

Index(['artwork id', 'artist id', 'first', 'last', 'nationality', 'year born',
       'year died', 'title', 'workyear modifier', 'workyear from',
       'workyear to', 'est lo usd', 'est hi usd', 'sale price usd', 'artist',
       'price'],
      dtype='object')

# Test on Consistency

In [4]:
Artist_name = "Pierre-Auguste Renoir"

In [7]:
artnet_2024_Renoir = artnet_2024[artnet_2024["artist"] == Artist_name].reset_index(drop=True)

In [13]:
artnet_2024_limit=copy.deepcopy(artnet_2024_Renoir)

In [14]:
artnet_2024_limit

,artwork id,artist id,first,last,nationality,year born,year died,title,workyear modifier,workyear from,workyear to,est lo usd,est hi usd,sale price usd,artist,price
0,41,14139,Pierre-Auguste,Renoir,French,1841,1919,Baigneuse,NaN,1888,1888.0,1.000000e+07,1.500000e+07,2.090250e+07,Pierre-Auguste Renoir,4.085286e+07
1,62,14139,Pierre-Auguste,Renoir,French,1841,1919,"Femme au peplum rouge, tête, bras",NaN,1895,NaN,8.695652e+04,1.217391e+05,1.920000e+05,Pierre-Auguste Renoir,2.987514e+05
2,149,14139,Pierre-Auguste,Renoir,French,1841,1919,Saule au bord d'une mare,CIRCA,1874,NaN,7.906674e+04,1.143112e+05,1.060334e+05,Pierre-Auguste Renoir,1.362007e+05
3,628,14139,Pierre-Auguste,Renoir,French,1841,1919,Compotier de fruits,CIRCA,1890,NaN,2.500000e+05,3.500000e+05,5.065000e+05,Pierre-Auguste Renoir,7.405878e+05
4,1184,14139,Pierre-Auguste,Renoir,French,1841,1919,Paysage,NaN,1917,NaN,8.150599e+04,1.191084e+05,9.526623e+04,Pierre-Auguste Renoir,1.367884e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2900,444330518,14139,Pierre-Auguste,Renoir,French,1841,1919,卡涅斯风景,NaN,1900,NaN,2.748952e+05,4.123428e+05,4.109683e+05,Pierre-Auguste Renoir,4.109683e+05
2901,444378044,14139,Pierre-Auguste,Renoir,French,1841,1919,Sur la plage à Bernval,CIRCA,1892,NaN,6.303183e+02,6.303183e+02,6.303183e+02,Pierre-Auguste Renoir,6.303183e+02
2902,444486964,14139,Pierre-Auguste,Renoir,French,1841,1919,Une Mère et Enfants,CIRCA,1912,NaN,6.000000e+02,9.000000e+02,3.250000e+02,Pierre-Auguste Renoir,3.250000e+02
2903,444878391,14139,Pierre-Auguste,Renoir,French,1841,1919,"Baigneuse debout, à mi-jambes",NaN,1910,NaN,4.760897e+02,4.760897e+02,4.760897e+02,Pierre-Auguste Renoir,4.760897e+02


In [25]:
with open("D:\\MissTiny\\GitHub\\Creativity_Chess\\Token_Key\\openai_api.txt", "r", encoding="utf-8") as file:
    openai_api = file.read()

In [26]:
openai_client = OpenAI(api_key=openai_api)

In [27]:
def annotation(i,content):
    row=[content["artwork id"],content["title"],content['first'],content['last'],content['workyear from'],content["nationality"]]

    instruction_message=fr"""You are an expert in artwork criticism, art history, and creativity evaluation.
    Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
    - Title: \'{content["title"]}\' 
    - Artist:{content['first']} {content['last']} ({content["nationality"]})
    - Year of Creation:{content['workyear from']} 
    
    Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
    Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
    - An artwork is creative only if it is new, valuable, and surprising compared to prior artworks
    
    OUTPUT FORMAT(around 150 words each)

    **Artistic Value: <High or Low>**
    <Your summary here>
    
    **Creativity: <Yes or No>**
    <Your summary here>
    """ 
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini-search-preview",      # or gpt-4o, gpt-4.1, gpt-4o-mini-SearchPreview
        web_search_options={},# <-- THIS ENABLES WEB SEARCH
        messages=[{
                "role": "user",
                "content": instruction_message,
            }
        ]
         
    )
    answer = response.choices[0].message.content
    artistic_answer = re.search(r"\*\*Artistic Value:\s*([^*]+)\*\*", answer)
    artistic_comment = re.search(r"\*\*Artistic Value:[^*]+\*\*\s*(.*?)\n\n\*\*Creativity", answer, re.S)

    creativity_answer = re.search(r"\*\*Creativity:\s*([^*]+)\*\*", answer)
    creativity_comment = re.search(r"\*\*Creativity:[^*]+\*\*\s*(.*)", answer, re.S)

    if not (artistic_answer and artistic_comment and creativity_answer and creativity_comment):
        row=[content["artwork id"], content["title"], content['first'], content['last'], content['workyear from'], content["nationality"],"","","",""]
        return i, row, answer
    
    result = {
        "artistic_value_answer": artistic_answer.group(1).strip() if artistic_answer else "",
        "artistic_value_comment": artistic_comment.group(1).strip() if artistic_comment else "",
        "creativity_answer": creativity_answer.group(1).strip() if creativity_answer else "",
        "creativity_comment": creativity_comment.group(1).strip() if creativity_comment else ""
    }

    row.append(result['artistic_value_answer'])
    row.append(result['artistic_value_comment'])
    row.append(result['creativity_answer'])
    row.append(result['creativity_comment'])
    return i, row, ""


In [28]:
# number_size= 5000
# number_size=df_raw.shape[0]
# range_start = 30000
range_start =0
# range_end = min(range_start+number_size,artnet_2024_Picasso.shape[0])
range_end = 1000
number_size = range_end - range_start
N = min(number_size, artnet_2024_limit.shape[0]-range_start)

In [29]:
error_output=[]
annotation_result = pd.DataFrame({
    "artwork id": np.full(number_size, np.nan, dtype=int),
    "title": [""] * number_size,
    "first": [""] * number_size,
    "last": [""] * number_size,
    "workyear from":np.full(number_size, np.nan, dtype=int),
    "nationality":[""] * number_size,
    "artistic_value_answer": [""] * number_size,
    "artistic_value_comment":[""] * number_size,
    "creativity_answer": [""] * number_size,
    "creativity_comment": [""] * number_size})
max_workers = 20  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(annotation, i,
                   {
                "artwork id": artnet_2024_limit.iloc[i]["artwork id"],
                "title": artnet_2024_limit.iloc[i]["title"],
                "first": artnet_2024_limit.iloc[i]["first"],
                "last": artnet_2024_limit.iloc[i]["last"],
                "workyear from": artnet_2024_limit.iloc[i]["workyear from"],
                "nationality": artnet_2024_limit.iloc[i]["nationality"]}
                ): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,result,error = fut.result()
        annotation_result.loc[i-range_start]  = result
        if error !="":
            error_output.append(error)
        completed += 1
        if completed % 50 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{len(artnet_2024_limit)}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(error_output)} errors")

C:\Users\MissTiny\anaconda3\envs\Creativity\Lib\site-packages\numpy\_core\numeric.py:353: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


2025-12-15 21:19:58: Parallel computation starts
2025-12-15 21:20:11: Completed 50/2905
2025-12-15 21:20:23: Completed 100/2905
2025-12-15 21:20:33: Completed 150/2905
2025-12-15 21:20:44: Completed 200/2905
2025-12-15 21:20:55: Completed 250/2905
2025-12-15 21:21:07: Completed 300/2905
2025-12-15 21:21:19: Completed 350/2905
2025-12-15 21:21:30: Completed 400/2905
2025-12-15 21:21:40: Completed 450/2905
2025-12-15 21:21:51: Completed 500/2905
2025-12-15 21:22:03: Completed 550/2905
2025-12-15 21:22:13: Completed 600/2905
2025-12-15 21:22:24: Completed 650/2905
2025-12-15 21:22:35: Completed 700/2905
2025-12-15 21:22:46: Completed 750/2905
2025-12-15 21:22:57: Completed 800/2905
2025-12-15 21:23:07: Completed 850/2905
2025-12-15 21:23:16: Completed 900/2905
2025-12-15 21:23:27: Completed 950/2905
2025-12-15 21:23:38: Completed 1000/2905
2025-12-15 21:23:38: Ends
2025-12-15 21:23:38: There are 0 errors


In [30]:
annotation_result.to_excel(f"comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx",index=False)

# Check and Fix

In [ ]:
empty_idx = annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist()

In [ ]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start")
count= 0
for error_index in empty_idx:
    if count%10 ==0:
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Current at {count}")
    content = {
                "artwork id": artnet_2024_limit.iloc[error_index]["artwork id"],
                "title": artnet_2024_limit.iloc[error_index]["title"],
                "first": artnet_2024_limit.iloc[error_index]["first"],
                "last": artnet_2024_limit.iloc[error_index]["last"],
                "workyear from": artnet_2024_limit.iloc[error_index]["workyear from"],
                "nationality": artnet_2024_limit.iloc[error_index]["nationality"]}
    _,result,error = annotation(error_index,content)
    annotation_result.loc[error_index,:] =result
    count+=1
    # if count ==10:
    #     break
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")

In [ ]:
annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist()

# Test on Prompt Sensitivity Code

In [22]:
with open("D:\\MissTiny\\GitHub\\Creativity_Chess\\Token_Key\\openai_api.txt", "r", encoding="utf-8") as file:
    openai_api = file.read()

In [81]:
row = artnet_2024_Picasso.iloc[1]

In [82]:
instruction_message=f"""You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'{row.title}\' 
- Artist:{row['first']} {row['last']} ({row.nationality})
- Year of Creation:{row['workyear from']} 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value: <High or Low>
<Your summary here>
Creativity: <Yes or No>
<Your summary here>
"""

In [83]:
print(instruction_message)

You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: 'Mousquetaire buste' 
- Artist:Pablo Picasso (Spanish)
- Year of Creation:1968 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value: <High or Low>
<Your summary here>
Creativity: <Yes or No>
<Your summary here>



In [84]:
openai_client = OpenAI(api_key=openai_api)

In [85]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini-search-preview",      # or gpt-4o, gpt-4.1, gpt-4o-mini-SearchPreview
    web_search_options={},# <-- THIS ENABLES WEB SEARCH
    messages=[{
            "role": "user",
            "content": instruction_message,
        }
    ]
     
)

In [86]:
answer = response.choices[0].message.content

In [87]:
print(answer)



**Artistic Value: High**

Pablo Picasso's 'Mousquetaire buste' (1968) exemplifies the artist's late-career mastery and inventive spirit. Painted during a period of convalescence in late 1965, Picasso revisited themes from classic literature, notably Alexandre Dumas' 'The Three Musketeers,' integrating them into his work. This series, including 'Mousquetaire buste,' is characterized by vibrant palettes, dynamic brushwork, and a sense of exuberance. The musketeer figure, adorned in elaborate costumes and often depicted with a pipe, serves as a vessel for Picasso to portray himself and comment on contemporary events. The painting's inventiveness and variety, along with its rich brushwork, contribute to its high artistic value. ([press.christies.com](https://press.christies.com/pablo-picassos-mousquetaire-a-la-pipe-to-highlight-christies-20th-century-evening-sale-in-new-york?utm_source=openai))

**Creativity: Yes**

'Mousquetaire buste' demonstrates Picasso's creativity through its innov

In [30]:
artistic_answer = re.search(r"\*\*Artistic Value:\s*([^*]+)\*\*", answer)
artistic_comment = re.search(r"\*\*Artistic Value:[^*]+\*\*\s*(.*?)\n\n\*\*Creativity", answer, re.S)

In [31]:
creativity_answer = re.search(r"\*\*Creativity:\s*([^*]+)\*\*", answer)
creativity_comment = re.search(r"\*\*Creativity:[^*]+\*\*\s*(.*)", answer, re.S)

In [32]:
result = {
    "artistic_value_answer": artistic_answer.group(1).strip() if artistic_answer else None,
    "artistic_value_comment": artistic_comment.group(1).strip() if artistic_comment else None,
    "creativity_answer": creativity_answer.group(1).strip() if creativity_answer else None,
    "creativity_comment": creativity_comment.group(1).strip() if creativity_comment else None
}

In [33]:
result

{'artistic_value_answer': 'High',
 'artistic_value_comment': 'Pablo Picasso\'s "Mousquetaire buste," painted in 1968, exemplifies the artist\'s late-career mastery and inventive spirit. This work is part of his renowned "musketeer" series, where he reimagines the 17th-century cavalier archetype, blending historical references with personal expression. The painting is characterized by vibrant colors, dynamic brushwork, and a sense of exuberance, reflecting Picasso\'s continued innovation and engagement with classical themes. ([press.christies.com](https://press.christies.com/pablo-picassos-mousquetaire-a-la-pipe-to-highlight-christies-20th-century-evening-sale-in-new-york?utm_source=openai))',
 'creativity_answer': 'Yes',
 'creativity_comment': '"Mousquetaire buste" demonstrates Picasso\'s creative prowess through its unique composition and technique. By revisiting the musketeer motif, he introduces a fresh perspective, infusing the figure with contemporary energy and personal symbolism

In [117]:
possible_prompts=[]

In [11]:
prompt1=f"""You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'{row.title}\' 
- Artist:{row['first']} {row['last']} ({row.nationality})
- Year of Creation:{row['workyear from']} 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value: <High or Low>
<Your summary here>
Creativity: <Yes or No>
<Your summary here>
"""

In [12]:
response1 = openai_client.chat.completions.create(
    model="gpt-4o-mini-search-preview",      # or gpt-4o, gpt-4.1, gpt-4o-mini-SearchPreview
    web_search_options={},# <-- THIS ENABLES WEB SEARCH
    messages=[{
            "role": "user",
            "content": prompt1,
        }
    ]
     
)

In [13]:
print(response1.choices[0].message.content)

**Artistic Value: High**

Pablo Picasso's 'Mousquetaire buste' (1968) exemplifies the artist's late-career mastery and inventive spirit. Executed with vibrant colors and dynamic brushwork, the painting portrays a musketeer—a recurring motif in Picasso's work during this period. The subject's elaborate attire and confident demeanor reflect a fusion of historical and personal symbolism, showcasing Picasso's ability to reinterpret classical themes with modern sensibilities. The painting's energetic composition and rich palette contribute to its status as a significant work in Picasso's oeuvre. ([press.christies.com](https://press.christies.com/pablo-picassos-mousquetaire-a-la-pipe-to-highlight-christies-20th-century-evening-sale-in-new-york?utm_source=openai))

**Creativity: Yes**

'Mousquetaire buste' demonstrates Picasso's innovative approach to composition and technique. By revisiting the musketeer theme, he infuses it with new life, blending elements of Cubism and Expressionism to cre

In [121]:
prompt2=f"""Act as an expert in artwork criticism, art history, and creativity evaluation.
Using web search, summarize in English the artistic value and creative significance of {row['first']} {row['last']}’s {row['workyear from']}  artwork “{row.title}”.
Note that an artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT: (~150 words each)
Artistic Value:
<text>

Creativity: <Yes or No>
<text>
"""

In [122]:
response2 = openai_client.chat.completions.create(
    model="gpt-4o-mini-search-preview",      # or gpt-4o, gpt-4.1, gpt-4o-mini-SearchPreview
    web_search_options={},# <-- THIS ENABLES WEB SEARCH
    messages=[{
            "role": "user",
            "content": prompt2,
        }
    ]
     
)

In [123]:
print(response2.choices[0].message.content)

**Artistic Value:**

Rogier van der Weyden's "Pietà," painted around 1441, is a masterpiece of Early Netherlandish art, renowned for its emotional depth and technical precision. The composition features the Virgin Mary cradling the lifeless body of Christ, with Saint John and Mary Magdalene present, set against a serene landscape illuminated by the soft light of dusk. The figures are meticulously detailed, with Christ's limp arms and fingers reflecting van der Weyden's acute observation of human anatomy. The balanced composition and harmonious color palette enhance the painting's emotional resonance, inviting viewers into a profound contemplation of the scene. This work exemplifies van der Weyden's mastery in conveying complex human emotions through art, solidifying its status as a significant contribution to the art historical canon. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Piet%C3%A0_%28van_der_Weyden%29?utm_source=openai))

**Creativity: Yes**

"Pietà" demonstrates van der 

In [126]:
prompt3=f"""ou are a general expert in artwork criticism, art history, and creativity evaluation.
Using web search, analyze the artwork:

- Title: '{row.title}'
- Artist: {row['first']} {row['last']} ({row.nationality})
- Year: {row['workyear from']}

Based ONLY on web search results, provide two ~150-word sections evaluating:
1. Artistic Value
2. Creativity

Note that an artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT:
Artistic Value:
<text>
Creativity: <Yes or No>
<text>
"""

In [127]:
response3 = openai_client.chat.completions.create(
    model="gpt-4o-mini-search-preview",      # or gpt-4o, gpt-4.1, gpt-4o-mini-SearchPreview
    web_search_options={},# <-- THIS ENABLES WEB SEARCH
    messages=[{
            "role": "user",
            "content": prompt3,
        }
    ]
     
)

In [128]:
prompt4=f"""Assume the role of a comprehensive art expert with deep knowledge of global art history, visual analysis, and creativity theory.
Using web search, gather authoritative scholarly commentary on:

- '{row.title}' by {row['first']} {row['last']} ({row.nationality}), created in {row['workyear from']}.

Based strictly on verified search results, summarize the artwork’s:
1. Artistic Value — aesthetic qualities, emotional resonance, technical execution, symbolism, and critical reception.
2. Creativity — an artwork is creative only if it is new, valuable, and surprising compared to prior artworks.

Write ~150 words per section, in English.

FORMAT:
Artistic Value:
<text>

Creativity: <Yes or No>
<text>
"""

In [129]:
response4 = openai_client.chat.completions.create(
    model="gpt-4o-mini-search-preview",      # or gpt-4o, gpt-4.1, gpt-4o-mini-SearchPreview
    web_search_options={},# <-- THIS ENABLES WEB SEARCH
    messages=[{
            "role": "user",
            "content": prompt4,
        }
    ]
     
)

In [130]:
print(response4.choices[0].message.content)

"Pietà" by Rogier van der Weyden, created around 1441, is a poignant depiction of the Virgin Mary cradling the lifeless body of Christ. The composition is marked by a harmonious balance, with the figures of Mary and Christ forming a unified, triangular shape that draws the viewer's eye toward the central scene. The artist's meticulous attention to detail is evident in the naturalistic rendering of Christ's limp body, with arms and fingers hanging lifelessly, showcasing van der Weyden's acute observation skills. The use of light and shadow enhances the three-dimensionality of the figures, adding depth and realism to the scene. Symbolically, the skull at the foreground alludes to Golgotha, the site of Christ's crucifixion, and may represent the fall of Adam, whose sin Christ came to redeem. The withered trees flanking the composition are drawn from medieval Passion narratives, signifying the desolation following Christ's death. The emotional intensity of the painting reflects the influen

In [131]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [132]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [133]:
texts=[response1.choices[0].message.content,response2.choices[0].message.content,response3.choices[0].message.content,response4.choices[0].message.content]

In [134]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
text_embeds = outputs.pooler_output
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
text_embeds=text_embeds.cpu().numpy()

In [135]:
pairwise = cosine_similarity(text_embeds)